# 🔗 Data Integration
## Merge Datasets for Unified Analysis

This notebook creates integrated datasets by merging:
- Price data with cost of cultivation
- Climate data with agriculture production
- Create master state/district-level analysis file

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW_PATH = Path('../data/raw/')
PROCESSED_PATH = Path('../data/processed/')
INTERMEDIATE_PATH = Path('../data/intermediate/')
INTERMEDIATE_PATH.mkdir(exist_ok=True)

print("Data Integration Pipeline Ready")

Data Integration Pipeline Ready


## 1. Load Cleaned Datasets

In [2]:
# Load all cleaned datasets
msp = pd.read_csv(PROCESSED_PATH / 'msp_cleaned.csv')
coc = pd.read_csv(PROCESSED_PATH / 'cost_of_cultivation_cleaned.csv')
retail = pd.read_csv(PROCESSED_PATH / 'retail_prices_cleaned.csv', parse_dates=['date'])
rainfall = pd.read_csv(PROCESSED_PATH / 'rainfall_state_cleaned.csv', parse_dates=['date'])
groundwater = pd.read_csv(PROCESSED_PATH / 'groundwater_cleaned.csv', parse_dates=['date'])
vulnerability = pd.read_csv(PROCESSED_PATH / 'vulnerability_state_cleaned.csv')

print("Loaded datasets:")
for name, df in [('MSP', msp), ('Cost of Cultivation', coc), ('Retail Prices', retail),
                 ('Rainfall', rainfall), ('Groundwater', groundwater), ('Vulnerability', vulnerability)]:
    print(f"  • {name}: {len(df):,} rows")

Loaded datasets:
  • MSP: 737 rows
  • Cost of Cultivation: 3,880 rows
  • Retail Prices: 2,104,788 rows
  • Rainfall: 204,876 rows
  • Groundwater: 550,850 rows
  • Vulnerability: 30 rows


---
## 2. Create Crop Mapping Table

In [3]:
# Map commodity names across datasets
CROP_MAPPING = {
    # MSP crop -> Retail commodity -> Cost of Cultivation crop
    'paddy': {
        'msp_names': ['Paddy - Common', "Paddy - Grade 'A'"],
        'retail_names': ['Rice'],
        'coc_names': ['Paddy']
    },
    'wheat': {
        'msp_names': ['Wheat'],
        'retail_names': ['Wheat'],
        'coc_names': ['Wheat']
    },
    'maize': {
        'msp_names': ['Maize'],
        'retail_names': [],
        'coc_names': ['Maize']
    },
    'gram': {
        'msp_names': ['Gram'],
        'retail_names': ['Gram Dal'],
        'coc_names': ['Gram']
    },
    'tur': {
        'msp_names': ['Arhar (Tur)'],
        'retail_names': ['Tur/Arhar Dal'],
        'coc_names': ['Tur (Arhar)']
    },
    'groundnut': {
        'msp_names': ['Groundnut'],
        'retail_names': ['Groundnut Oil'],
        'coc_names': ['Groundnut']
    },
    'cotton': {
        'msp_names': ['Cotton Medium Staple', 'Cotton Long Staple'],
        'retail_names': [],
        'coc_names': ['Cotton']
    },
    'sugarcane': {
        'msp_names': ['Sugarcane'],
        'retail_names': ['Sugar'],
        'coc_names': ['Sugarcane']
    }
}

def normalize_crop_name(name, dataset_type='msp'):
    """Normalize crop/commodity name to standard key."""
    name = str(name).strip().lower()
    
    for standard_name, mappings in CROP_MAPPING.items():
        if dataset_type == 'msp':
            check_list = [n.lower() for n in mappings['msp_names']]
        elif dataset_type == 'retail':
            check_list = [n.lower() for n in mappings['retail_names']]
        elif dataset_type == 'coc':
            check_list = [n.lower() for n in mappings['coc_names']]
        else:
            check_list = []
        
        if name in check_list:
            return standard_name
    
    return None  # No match found

print("Crop mapping table created")

Crop mapping table created


---
## 3. Integration 1: Price Analysis Dataset
Merge MSP + Cost of Cultivation + Retail Prices

In [4]:
# Add normalized crop names
msp['crop_key'] = msp['crop'].apply(lambda x: normalize_crop_name(x, 'msp'))
coc['crop_key'] = coc['crop_name'].apply(lambda x: normalize_crop_name(x, 'coc'))

# Filter to matching crops only
msp_matched = msp[msp['crop_key'].notna()].copy()
coc_matched = coc[coc['crop_key'].notna()].copy()

print(f"MSP records with matching crops: {len(msp_matched):,}")
print(f"Cost of Cultivation records with matching crops: {len(coc_matched):,}")

MSP records with matching crops: 145
Cost of Cultivation records with matching crops: 2,028


In [5]:
# Aggregate MSP by crop and year
msp_agg = msp_matched.groupby(['year_start', 'crop_key', 'season']).agg({
    'min_support_price': 'mean'
}).reset_index()

# Aggregate Cost of Cultivation by state, crop, year
coc_agg = coc_matched.groupby(['year_start', 'state_name', 'crop_key', 'crop_type']).agg({
    'cul_cost_c2': 'mean',
    'main_product_value': 'mean',
    'derived_yield': 'mean',
    'profit_margin_pct': 'mean'
}).reset_index()

print(f"Aggregated MSP: {len(msp_agg):,} records")
print(f"Aggregated CoC: {len(coc_agg):,} records")

Aggregated MSP: 120 records
Aggregated CoC: 2,023 records


In [6]:
# Merge MSP with Cost of Cultivation
price_analysis = pd.merge(
    coc_agg,
    msp_agg,
    on=['year_start', 'crop_key'],
    how='left'
)

# Calculate MSP coverage ratio
price_analysis['msp_coverage_ratio'] = (
    price_analysis['min_support_price'] / price_analysis['cul_cost_c2']
).round(3)

# MSP vs Market price comparison
price_analysis['msp_vs_market_pct'] = (
    (price_analysis['main_product_value'] - price_analysis['min_support_price']) / 
    price_analysis['min_support_price'] * 100
).round(2)

print(f"\nPrice Analysis Dataset: {len(price_analysis):,} records")
price_analysis.head(10)


Price Analysis Dataset: 2,023 records


,year_start,state_name,crop_key,crop_type,cul_cost_c2,main_product_value,derived_yield,profit_margin_pct,season,min_support_price,msp_coverage_ratio,msp_vs_market_pct
0,1996,Andhra Pradesh,cotton,Fiber Crops,23776.29,22103.81,12.95,-7.57,NaN,NaN,NaN,NaN
1,1996,Andhra Pradesh,groundnut,Oilseeds,12172.16,10326.37,9.03,-17.87,NaN,NaN,NaN,NaN
2,1996,Andhra Pradesh,maize,Cereals,12523.78,10136.62,23.07,-23.55,NaN,NaN,NaN,NaN
3,1996,Andhra Pradesh,paddy,Cereals,20937.09,20166.02,47.04,-3.82,NaN,NaN,NaN,NaN
4,1996,Assam,paddy,Cereals,8616.85,8659.21,21.01,0.49,NaN,NaN,NaN,NaN
5,1996,Bihar,gram,Pulses,7453.60,8754.98,7.33,14.86,NaN,NaN,NaN,NaN
6,1996,Bihar,maize,Cereals,8687.82,9528.31,22.61,8.82,NaN,NaN,NaN,NaN
7,1996,Bihar,paddy,Cereals,9395.03,8876.38,21.43,-5.84,NaN,NaN,NaN,NaN
8,1996,Bihar,wheat,Cereals,10893.53,11791.92,22.69,7.62,NaN,NaN,NaN,NaN
9,1996,Gujarat,cotton,Fiber Crops,13316.92,17074.38,9.50,22.01,NaN,NaN,NaN,NaN


In [7]:
# Save
price_analysis.to_csv(INTERMEDIATE_PATH / 'price_analysis_integrated.csv', index=False)
print(f"Saved: {INTERMEDIATE_PATH / 'price_analysis_integrated.csv'}")

Saved: ..\data\intermediate\price_analysis_integrated.csv


---
## 4. Integration 2: State-level Climate-Agriculture Dataset

In [8]:
# Aggregate rainfall by state and year
rainfall_yearly = rainfall.groupby(['state_name', 'year']).agg({
    'actual': 'sum',
    'normal': 'sum',
    'deviation': 'mean'
}).reset_index()
rainfall_yearly.columns = ['state_name', 'year', 'annual_rainfall', 'normal_rainfall', 'avg_deviation']

print(f"Yearly rainfall summary: {len(rainfall_yearly):,} records")
rainfall_yearly.head()

Yearly rainfall summary: 576 records


,state_name,year,annual_rainfall,normal_rainfall,avg_deviation
0,Andaman & Nicobar,2009,0.0,2909.48,NaN
1,Andaman & Nicobar,2010,0.0,2909.48,NaN
2,Andaman & Nicobar,2011,0.0,2909.48,NaN
3,Andaman & Nicobar,2012,0.0,2909.48,NaN
4,Andaman & Nicobar,2013,0.0,2909.48,NaN


In [9]:
# Aggregate groundwater by state and year
gw_yearly = groundwater.groupby(['state_name', 'year']).agg({
    'currentlevel': 'mean',
    'level_diff': 'mean'
}).reset_index()
gw_yearly.columns = ['state_name', 'year', 'avg_water_depth', 'avg_level_change']

print(f"Yearly groundwater summary: {len(gw_yearly):,} records")
gw_yearly.head()

Yearly groundwater summary: 285 records


,state_name,year,avg_water_depth,avg_level_change
0,Andaman & Nicobar,2013,0.953188,-0.491304
1,Andaman & Nicobar,2014,2.179659,0.182000
2,Andaman & Nicobar,2015,1.702809,-0.186067
3,Andaman & Nicobar,2016,1.896754,0.005079
4,Andaman & Nicobar,2017,1.590690,0.018448


In [10]:
# Merge rainfall and groundwater
climate_state = pd.merge(
    rainfall_yearly,
    gw_yearly,
    on=['state_name', 'year'],
    how='outer'
)

# Add vulnerability indicators
climate_state = pd.merge(
    climate_state,
    vulnerability[['state_name', 'climate_vul_in', 'vulnerability_normalized', 'risk_category']],
    on='state_name',
    how='left'
)

print(f"\nClimate-State Dataset: {len(climate_state):,} records")
climate_state.head(10)


Climate-State Dataset: 576 records


,state_name,year,annual_rainfall,normal_rainfall,avg_deviation,avg_water_depth,avg_level_change,climate_vul_in,vulnerability_normalized,risk_category
0,Andaman & Nicobar,2009,0.0,2909.48,NaN,NaN,NaN,NaN,NaN,NaN
1,Andaman & Nicobar,2010,0.0,2909.48,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman & Nicobar,2011,0.0,2909.48,NaN,NaN,NaN,NaN,NaN,NaN
3,Andaman & Nicobar,2012,0.0,2909.48,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman & Nicobar,2013,0.0,2909.48,NaN,0.953188,-0.491304,NaN,NaN,NaN
5,Andaman & Nicobar,2014,0.0,2909.48,NaN,2.179659,0.182000,NaN,NaN,NaN
6,Andaman & Nicobar,2015,0.0,2909.48,NaN,1.702809,-0.186067,NaN,NaN,NaN
7,Andaman & Nicobar,2016,0.0,2909.48,NaN,1.896754,0.005079,NaN,NaN,NaN
8,Andaman & Nicobar,2017,0.0,2909.48,NaN,1.590690,0.018448,NaN,NaN,NaN
9,Andaman & Nicobar,2018,0.0,2909.48,NaN,1.120842,-0.104208,NaN,NaN,NaN


In [11]:
# Save
climate_state.to_csv(INTERMEDIATE_PATH / 'climate_state_integrated.csv', index=False)
print(f"Saved: {INTERMEDIATE_PATH / 'climate_state_integrated.csv'}")

Saved: ..\data\intermediate\climate_state_integrated.csv


---
## 5. Integration 3: Master State Analysis Dataset

In [12]:
# Aggregate price analysis by state
price_state = price_analysis.groupby('state_name').agg({
    'cul_cost_c2': 'mean',
    'main_product_value': 'mean',
    'profit_margin_pct': 'mean',
    'msp_coverage_ratio': 'mean'
}).reset_index()
price_state.columns = ['state_name', 'avg_cultivation_cost', 'avg_product_value', 
                       'avg_profit_margin', 'avg_msp_coverage']

print(f"Price summary by state: {len(price_state):,} records")

Price summary by state: 20 records


In [13]:
# Latest year climate summary
latest_year = climate_state['year'].max()
climate_latest = climate_state[climate_state['year'] == latest_year].copy()

# Merge all state-level data
master_state = pd.merge(
    vulnerability,
    price_state,
    on='state_name',
    how='left'
)

master_state = pd.merge(
    master_state,
    climate_latest[['state_name', 'annual_rainfall', 'avg_deviation', 'avg_water_depth']],
    on='state_name',
    how='left'
)

print(f"\nMaster State Dataset: {len(master_state):,} records")
print(f"Columns: {list(master_state.columns)}")
master_state.head(10)


Master State Dataset: 30 records
Columns: ['state_name', 'state_code', 'climate_vul_in', 'yield_variability', 'irrigated_area', 'rainfed_agriculture', 'poverty_rate', 'women_workforce', 'employed_under_mgnrega', 'road_rail_density', 'vulnerability_normalized', 'risk_category', 'avg_cultivation_cost', 'avg_product_value', 'avg_profit_margin', 'avg_msp_coverage', 'annual_rainfall', 'avg_deviation', 'avg_water_depth']


,state_name,state_code,climate_vul_in,yield_variability,irrigated_area,rainfed_agriculture,poverty_rate,women_workforce,employed_under_mgnrega,road_rail_density,vulnerability_normalized,risk_category,avg_cultivation_cost,avg_product_value,avg_profit_margin,avg_msp_coverage,annual_rainfall,avg_deviation,avg_water_depth
0,Andhra Pradesh,28,0.510,0.08,46.94,0.53,9.20,36.16,47.0,1.11,0.356863,Medium,53667.577229,55884.137169,-2.396928,0.029376,4.59,-34.493418,NaN
1,Arunachal Pradesh,12,0.594,0.18,24.89,0.75,34.67,35.44,14.0,0.51,0.686275,High,NaN,NaN,NaN,NaN,150.19,-44.104051,NaN
2,Assam,18,0.620,0.15,10.47,0.90,31.98,22.46,22.0,4.31,0.788235,High,29013.781923,23615.352692,-19.169231,0.037125,46.21,-40.010633,NaN
3,Bihar,10,0.614,0.20,56.59,0.43,33.74,19.07,34.0,2.23,0.764706,High,27580.648729,32754.437627,8.635085,0.058278,19.46,-21.794304,NaN
4,Chhattisgarh,22,0.623,0.16,31.32,0.69,39.93,39.70,32.0,0.72,0.800000,High,26165.524054,25958.281486,-8.216892,0.072099,36.17,20.634684,NaN
5,Goa,30,0.434,0.08,30.23,0.70,5.09,21.92,23.0,6.09,0.058824,Low,NaN,NaN,NaN,NaN,3.67,72.777778,NaN
6,Gujarat,24,0.562,0.13,41.09,0.59,16.63,23.38,35.0,0.92,0.560784,Medium,40174.529212,46150.588048,6.723390,0.039823,3.31,15.217391,NaN
7,Haryana,6,0.463,0.08,84.44,0.16,11.16,17.79,28.0,1.93,0.172549,Low,48998.608862,63538.975041,12.423008,0.042942,37.05,23.037595,NaN
8,Himachal Pradesh,2,0.486,0.10,20.55,0.79,8.06,44.82,42.0,1.16,0.262745,Low,26684.231324,20411.342059,-37.999412,0.044672,215.20,-19.747468,NaN
9,Jammu & Kashmir,1,0.550,0.10,43.67,0.56,10.35,19.11,36.0,0.29,0.513725,Medium,NaN,NaN,NaN,NaN,198.06,-36.964430,NaN


In [14]:
# Calculate composite vulnerability score
# Normalize each component to 0-1 scale
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

master_state['water_stress_norm'] = normalize(master_state['avg_water_depth'].fillna(master_state['avg_water_depth'].median()))
master_state['rainfall_risk_norm'] = normalize(master_state['avg_deviation'].abs().fillna(0))
master_state['income_stress_norm'] = normalize(-master_state['avg_profit_margin'].fillna(0))  # Lower profit = higher stress

# Composite score (weighted average)
master_state['composite_vulnerability'] = (
    master_state['vulnerability_normalized'].fillna(0) * 0.4 +
    master_state['water_stress_norm'].fillna(0) * 0.25 +
    master_state['rainfall_risk_norm'].fillna(0) * 0.20 +
    master_state['income_stress_norm'].fillna(0) * 0.15
).round(3)

# Risk ranking
master_state = master_state.sort_values('composite_vulnerability', ascending=False)
master_state['vulnerability_rank'] = range(1, len(master_state) + 1)

master_state[['state_name', 'composite_vulnerability', 'vulnerability_rank', 'risk_category']].head(15)

,state_name,composite_vulnerability,vulnerability_rank,risk_category
10,Jharkhand,0.591,1,High
20,Odisha,0.534,2,High
29,West Bengal,0.507,3,High
2,Assam,0.469,4,High
18,Mizoram,0.421,5,High
4,Chhattisgarh,0.412,6,High
27,Uttar Pradesh,0.393,7,Medium
1,Arunachal Pradesh,0.377,8,High
3,Bihar,0.350,9,High
9,Jammu & Kashmir,0.298,10,Medium


In [15]:
# Save master dataset
master_state.to_csv(INTERMEDIATE_PATH / 'master_state_analysis.csv', index=False)
print(f"\n✅ Saved: {INTERMEDIATE_PATH / 'master_state_analysis.csv'}")


✅ Saved: ..\data\intermediate\master_state_analysis.csv


---
## 6. Integration Summary

In [16]:
# List all integrated files
integrated_files = list(INTERMEDIATE_PATH.glob('*.csv'))

print("\n📁 INTEGRATED DATASETS")
print(f"Location: {INTERMEDIATE_PATH.absolute()}")
print("="*60)

for f in integrated_files:
    df = pd.read_csv(f)
    print(f"\n📊 {f.name}")
    print(f"   Rows: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Size: {f.stat().st_size / 1024:.1f} KB")


📁 INTEGRATED DATASETS
Location: c:\Users\Asus\OneDrive\Documents\Project-2\notebooks\..\data\intermediate

📊 climate_state_integrated.csv
   Rows: 576
   Columns: 10
   Size: 52.8 KB

📊 master_state_analysis.csv
   Rows: 30
   Columns: 24
   Size: 6.3 KB

📊 price_analysis_integrated.csv
   Rows: 2,023
   Columns: 12
   Size: 157.9 KB


---
## Next Steps

With integrated datasets ready, you can now:

1. **Vulnerability Analysis**: Use `master_state_analysis.csv` to identify high-risk states
2. **Price-Climate Correlation**: Analyze how rainfall impacts crop prices
3. **Profitability Assessment**: Compare MSP with actual market prices and cultivation costs
4. **Build Dashboards**: Create interactive visualizations using the integrated data

In [17]:
print("\n✅ Data Integration Complete!")


✅ Data Integration Complete!
